In [7]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.callbacks import StreamingStdOutCallbackHandler

import os

load_dotenv()

llm = ChatOpenAI(
    base_url = os.getenv("OPENAI_BASE_URL"),
    api_key = os.getenv("OPENAI_API_KEY"), 
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
    temperature = 0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

In [10]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.storage import LocalFileStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=50,
)

loader = TextLoader("./genshinImpact.txt")
docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings(
    base_url=os.getenv("OPENAI_EMBEDDING_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_EMBEDDING_MODEL"),
)

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up. \n{context}"),
    ("human", "{question}"),
])

chain = (
    {
    "context": retriever,
    "question": RunnablePassthrough(),
    } | prompt | llm
)

chain.invoke("Who is the main character in Genshin Impact?").content
print("\n")
chain.invoke("What does Archons mean?").content
print("\n")
chain.invoke("What is the name of the world in Genshin Impact").content

The main playable protagonist in Genshin Impact is called **the Traveler**.  

You play as the Traveler, who is searching for their lost sibling and exploring the world of Teyvat while getting involved in the affairs of its different nations and Archons.

In the context of Genshin Impact, **Archons** are the gods who rule over each region of the world, Teyvat.

- Each major nation (like Mondstadt, Liyue, Inazuma) is associated with an **element** (Anemo, Geo, Electro, etc.) and is **governed by an Archon** of that element.  
- Archons are central to the story and the deeper mysteries of the world and its powers.

Outside of the game, “archon” is a word that can mean a ruler or high official, which is similar to how it’s used in Genshin Impact.

The world in Genshin Impact is called **Teyvat**.

'The world in Genshin Impact is called **Teyvat**.'